In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
import glob
import json
import numpy as np
import anndata as ad
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

import sys
sys.path.append("/code")
import bg_spatial_plots.plot_spatial_histograms as histplot
from plot_confusion_matrix import plot_confusion_matrix

%matplotlib inline

In [3]:
adata_human_tax = ad.read_h5ad('/data/human_HMBA_BG_AIT_w_adj_MNH_custom/Human_HMBA_basalganglia_AIT_with_adj_types_MNH_custom.h5ad')


In [7]:
adata_human_tax.obs.columns.tolist()

['sample_id',
 'cell_barcode',
 'umi.counts',
 'ar_id',
 'library_prep',
 'barcodes',
 'gene_counts_0',
 'gene_counts_1',
 'gene_counts_4',
 'gene_counts_8',
 'gene_counts_16',
 'gene_counts_32',
 'gene_counts_64',
 'doublet_score',
 'exclude',
 'exclude2',
 'cell_member',
 'exp_component_name',
 'cell_prep_type',
 'barcoded_cell_sample_label',
 'expc_cell_capture',
 'ar_dir',
 'ocs_fastq_file_id',
 'ocs_alignment_file_id',
 'popqc',
 'genome',
 'pipeline_version',
 'estimated_number_of_cells',
 'feature_linkages_detected',
 'linked_genes',
 'linked_peaks',
 'atac_confidently_mapped_read_pairs',
 'atac_fraction_of_genome_in_peaks',
 'atac_fraction_of_high_quality_fragments_in_cells',
 'atac_fraction_of_high_quality_fragments_overlapping_tss',
 'atac_fraction_of_high_quality_fragments_overlapping_peaks',
 'atac_fraction_of_transposition_events_in_peaks_in_cells',
 'atac_mean_raw_read_pairs_per_cell',
 'atac_median_high_quality_fragments_per_cell',
 'atac_non_nuclear_read_pairs',
 'atac_

In [14]:
subgroup_cluster_dict = {
    'GM STR': ['h-7', 'h-14', 'h-31'],
    'GM exSTR': [
        'h-145',
        'h-146',
        'h-149',
        'h-150',
        'h-152',
        'h-159',
        'h-160',
        'h-450',
        'h-452'
    ],
    'WM': ['h-28', 'h-227', 'h-230', 'h-231', 'h-472', 'h-479', 'h-485'],
    'Mixed': ['h-143', 'h-232']
}

# replace h- with Human-
subgroup_cluster_dict = {k: [c.replace('h-', 'Human-') for c in v] for k, v in subgroup_cluster_dict.items()}
subgroup_cluster_dict

# invert the dictionary to get cluster to subgroup mapping
cluster_subgroup_dict = {c: k for k, v in subgroup_cluster_dict.items() for c in v}
cluster_subgroup_dict

{'Human-7': 'GM STR',
 'Human-14': 'GM STR',
 'Human-31': 'GM STR',
 'Human-145': 'GM exSTR',
 'Human-146': 'GM exSTR',
 'Human-149': 'GM exSTR',
 'Human-150': 'GM exSTR',
 'Human-152': 'GM exSTR',
 'Human-159': 'GM exSTR',
 'Human-160': 'GM exSTR',
 'Human-450': 'GM exSTR',
 'Human-452': 'GM exSTR',
 'Human-28': 'WM',
 'Human-227': 'WM',
 'Human-230': 'WM',
 'Human-231': 'WM',
 'Human-472': 'WM',
 'Human-479': 'WM',
 'Human-485': 'WM',
 'Human-143': 'Mixed',
 'Human-232': 'Mixed'}

In [15]:
for cluster in adata_human_tax.obs['cluster_alias'].unique():
    if cluster not in cluster_subgroup_dict:
        group = adata_human_tax.obs['Group'][adata_human_tax.obs['cluster_alias'] == cluster].iloc[0]
        print(group)
        cluster_subgroup_dict[cluster] = group

VLMC
Oligo OPALIN
Microglia
STRv D1 MSN
Oligo PLEKHG1
STRd D2 Matrix MSN
OPC
STRd D1 Matrix MSN
STRv D1 MSN
STRd D2 Matrix MSN
STRv D1 MSN
STRv D1 MSN
STRd D2 Striosome MSN
STRv D2 MSN
STRd D2 Matrix MSN
Oligo OPALIN
Oligo PLEKHG1
STR TAC3-PLPP4 GABA
STR D1D2 Hybrid MSN
ImAstro
STRv D2 MSN
GPe SOX6-CTXND1 GABA
Microglia
STRd D2 Striosome MSN
AMY-SLEA-BNST GABA
STRd D1 Matrix MSN
STRv D1 NUDAP MSN
STRd D1 Striosome MSN
STRd D1 Striosome MSN
Microglia
STR FS PTHLH-PVALB GABA
STRv D1 MSN
STR D1D2 Hybrid MSN
STRd D2 StrioMat Hybrid MSN
STRd D2 Striosome MSN
STRv D1 NUDAP MSN
Microglia
Pericyte
STR-BF TAC3-PLPP4-LHX8 GABA
STRv D2 MSN
STRd D1 Striosome MSN
OPC
STRd D1 Striosome MSN
STRd D2 Striosome MSN
STRv D2 MSN
STRd D1 Matrix MSN
STRv D2 MSN
STR SST-CHODL GABA
STRv D1 NUDAP MSN
ImOligo
STR SST-RSPO2 GABA
GPe MEIS2-SOX6 GABA
COP
OT D1 ICj
STRv D1 NUDAP MSN
STRv D2 MSN
STRv D1 NUDAP MSN
ZI-HTH GABA
STR D1D2 Hybrid MSN
STRv D1 MSN
STRd D1 Matrix MSN
Endo
GPe MEIS2-SOX6 GABA
Microglia
STRd D

In [16]:
adata_human_tax.obs['SubGroup'] = adata_human_tax.obs['cluster_alias'].map(cluster_subgroup_dict)

In [19]:
adata_human_tax.write_h5ad('/results/Human_HMBA_basalganglia_AIT_with_adj_types_MNH_custom_w_Astro_SubGroups.h5ad', compression='gzip')

# Make an astrocyte-only spatial dataset for re-mapping for QC

In [3]:
adata_spatial = ad.read_h5ad('/scratch/astrocytes/human_spatial_astrocytes_extended_mmc_results_UMAP_colors.h5ad')
adata_spatial

AnnData object with n_obs × n_vars = 641913 × 299
    obs: 'x', 'y', 'brain_section_label', 'brain_section_barcode', 'slab', 'block', 'set', 'z_order', 'total_counts', 'total_counts_genes', 'n_genes_by_counts', 'total_counts_Blank', 'pct_counts_Blank', 'genes_filter', 'counts_filter', 'blanks_filter', 'qc_pass', 'doublets_filter', 'qc_pass_and_singlet', 'doublet_singlet_score_diff', 'doublet_diff_threshold', 'hierarchy_consistent', 'Neighborhood', 'Neighborhood_bootstrapping_probability', 'Neighborhood_correlation_coefficient', 'Class', 'Class_bootstrapping_probability', 'Class_correlation_coefficient', 'Subclass', 'Subclass_bootstrapping_probability', 'Subclass_correlation_coefficient', 'Group', 'Group_bootstrapping_probability', 'Group_correlation_coefficient', 'cluster_id', 'cluster_id_bootstrapping_probability', 'cluster_id_correlation_coefficient', 'Neighborhood_entropy', 'Class_entropy', 'Subclass_entropy', 'Group_entropy', 'cluster_id_entropy', 'STAligner_mclust_res_10_knn_15', 